# Set up the Fabric policy - existing workspaces

Use this notebook when your **three workspaces already exist**. It finds them, creates a **policy set** in the
**policy-store** workspace, adds three **rules** to it, and **turns it on** for the capacity of the **Data
Engineering** and **Data Engineering + AI** workspaces. The steps are explained just before the cell that runs them.

| Workspace | Setting | Purpose |
| --- | --- | --- |
| **Policy store** | `POLICY_STORE_WORKSPACE` | Stores the policy set. It isn't restricted. |
| **Data Engineering** | `DATA_ENGINEERING_WORKSPACE` | Can create only Lakehouse, Data pipeline and Notebook. |
| **Data Engineering + AI** | `DATA_ENGINEERING_AI_WORKSPACE` | Can create the same items, plus Data agent. |

| Rule | Applies to | Allows |
| --- | --- | --- |
| 1 | Data Engineering and Data Engineering + AI workspaces | Lakehouse, Data pipeline, Notebook |
| 2 | Data Engineering + AI workspace | Data agent |
| 3 | Every other workspace on the capacity | Everything |

So, after up to 15 minutes, the **Data Engineering** workspace can't create a Data agent, the **Data Engineering +
AI** workspace can, and every other workspace on the capacity works as before.

### Before you start

* Import this notebook **and** `policy_setup_functions.ipynb` into the **same workspace**.
* Your Fabric administrator has turned on the **Policies** tenant setting for you.
* Your role on each workspace:

  | Workspace | Role you need | Why |
  | --- | --- | --- |
  | **Policy store** | **Contributor**, **Member** or **Admin** | The policy set is created in it. |
  | **Data Engineering** and **Data Engineering + AI** | **Viewer** or higher | The notebook only finds them and reads their capacity; it doesn't change them. |

* To turn the policy set on, you need the **Capacity Admin** role. Without it, steps 1-3 still run, and step 4
  prints a message for your Capacity Admin.

### How to run

Type your workspace names in the **Settings** cell, and then select **Run all**. Each step says what it did. If
something goes wrong, the notebook stops and tells you what to do. It's safe to run again.

## Settings - type your workspace names here

In [ ]:
POLICY_STORE_WORKSPACE = "Policy-Admin-Store"                 # Policy store: stores the policy set
DATA_ENGINEERING_WORKSPACE = "Policy-DataEngineering"         # Data Engineering: Lakehouse, Data pipeline, Notebook
DATA_ENGINEERING_AI_WORKSPACE = "Policy-DataEngineering-AI"   # Data Engineering + AI: can also create a Data agent

POLICY_SET_NAME = "Fabric_ItemCreation_Governance"   # no spaces

## Load the setup functions

Loads the notebook **policy_setup_functions**, which has the code for every step below. It must be in the **same
workspace** as this notebook. If this cell says the notebook can't be found, import `policy_setup_functions.ipynb` into this
workspace (keep its name), and run again.

In [ ]:
%run policy_setup_functions

## Set up the policy - steps 1 to 4

The cell below runs all four steps, in order. Each step prints what it did. If a step fails, the notebook stops
there and tells you what to do.

| Step | What it does | Function |
| --- | --- | --- |
| 1. Find the workspaces | Finds the three workspaces, checks your role on the **policy-store** workspace (**Contributor** or higher), and checks that the **Data Engineering** and **Data Engineering + AI** workspaces are on the **same** Fabric capacity and that it's running. It also tells you whether you're a Capacity Admin. | `find_existing_workspaces` |
| 2. Create the policy set | Creates the policy set in the **policy-store** workspace. It stays **off** until step 4. If it already exists, it's reused. | `create_policy_set` |
| 3. Add the rules | Sets the three rules from the table at the top. Running it again replaces them, so they're never duplicated. | `add_rules` |
| 4. Turn on the policy set | Turns it on for the capacity. Needs the **Capacity Admin** role; without it, the policy set and rules stay saved (but off), and you get a message to send to your Capacity Admin. | `turn_on_policy_set` |

The functions are in `policy_setup_functions`, and the rules are in its **Rules** section.

In [ ]:
policy_store_ws, data_eng_ws, data_eng_ai_ws, capacity = find_existing_workspaces(   # Step 1
    POLICY_STORE_WORKSPACE, DATA_ENGINEERING_WORKSPACE, DATA_ENGINEERING_AI_WORKSPACE)
policy_set = create_policy_set(policy_store_ws, POLICY_SET_NAME)                     # Step 2
add_rules(policy_store_ws, policy_set, data_eng_ws, data_eng_ai_ws)                  # Step 3
turn_on_policy_set(policy_store_ws, policy_set, capacity)                            # Step 4

## Try it

After up to 15 minutes:

| Where | Create a... | Result |
| --- | --- | --- |
| Data Engineering | Data agent | Blocked |
| Data Engineering + AI | Data agent | Allowed |
| Data Engineering or Data Engineering + AI | Lakehouse, Data pipeline or Notebook | Allowed |
| Data Engineering or Data Engineering + AI | Anything else, such as a Warehouse | Blocked |
| Policy store, or any other workspace | Anything | Allowed |